# Geotech Foundations from a Boring Log

An invented site: one boring through 20 ft of medium clay into a dense sand,
water table at 12 ft.  From that single `civilpy.geotech` boring we run the
whole foundation ladder — SPT corrections, a spread footing (bearing +
settlement), a driven pile, and a drilled shaft.

## Build the boring

In practice borings come from `civilpy.geotech.boring_io.parse_diggs` (DIGGS
XML) or `read_pdf_log`; here we assemble one by hand.  Each `SPTResult`
carries its drive increments (the last two sum to the field N), and each
`GradingResult` a particle-size curve — fines content is what steers the
capacity methods between the clay (alpha, 9Su) and sand (Meyerhof, beta)
equations.

In [1]:
from civilpy.geotech.boring import (
    Borehole,
    DriveIncrement,
    GradingPoint,
    GradingResult,
    SPTResult,
)


def spt(depth, n):
    return SPTResult(
        depth, (DriveIncrement(4), DriveIncrement(n // 2), DriveIncrement(n - n // 2))
    )


def grading(depth, fines):
    return GradingResult(depth, (GradingPoint(2.0, 100), GradingPoint(0.075, fines)))


clay_depths = [2.5, 5.0, 7.5, 10.0, 12.5, 15.0, 17.5, 20.0]
sand_depths = [22.5, 25.0, 27.5, 30.0, 32.5, 35.0, 37.5, 40.0, 42.5, 45.0]

boring = Borehole(
    boring_id="B-201",
    total_depth_ft=45.0,
    water_strike_depth_ft=12.0,
    spt=[spt(d, 8) for d in clay_depths] + [spt(d, 32) for d in sand_depths],
    grading=[grading(d, 78) for d in clay_depths]
    + [grading(d, 9) for d in sand_depths],
)
print(
    boring.boring_id,
    f"to {boring.total_depth_ft} ft, water at {boring.water_strike_depth_ft} ft",
)
print("field N values:", [s.n_value for s in boring.spt])

B-201 to 45.0 ft, water at 12.0 ft
field N values: [8, 8, 8, 8, 8, 8, 8, 8, 32, 32, 32, 32, 32, 32, 32, 32, 32, 32]


## SPT corrections and correlations

`civilpy.geotech.spt` carries the standard chain: energy-correct the field N
to N60, normalize for overburden to (N1)60, then correlate to friction angle
and relative density in the sand:

In [2]:
from civilpy.geotech import spt as spt_mod

gamma = 120.0  # rough average unit weight, pcf
print(" depth   N   N60  (N1)60   phi    Dr")
for s in boring.spt[8:]:          # the sand below 20 ft
    n60 = float(s.n_value)         # assume the rig delivers ~60% energy
    sigma_v = gamma * s.depth_ft - 62.4 * max(0.0, s.depth_ft - 12.0)
    n1 = spt_mod.n1_60(n60, sigma_v)
    phi = spt_mod.friction_angle_from_n(n1)
    dr = spt_mod.relative_density_from_n(n1)
    print(
        f"{s.depth_ft:5.1f}  {s.n_value:3d}  {n60:4.0f}  {n1:6.1f}  {phi:5.1f}  "
        f"{dr:5.0%}"
    )

 depth   N   N60  (N1)60   phi    Dr
 22.5   32    32    31.6   45.2    73%
 25.0   32    32    30.6   44.7    71%
 27.5   32    32    29.6   44.3    70%
 30.0   32    32    28.8   44.0    69%
 32.5   32    32    28.0   43.6    68%
 35.0   32    32    27.2   43.3    67%
 37.5   32    32    26.5   43.0    67%
 40.0   32    32    25.9   42.8    66%
 42.5   32    32    25.3   42.5    65%
 45.0   32    32    24.8   42.3    64%


## Spread footing — bearing capacity

An 8 ft x 10 ft footing bearing at 4 ft in the clay.  The undrained
correlation Su ≈ 92·N60 psf feeds the general bearing-capacity equation
(AASHTO LRFD 10.6.3.1.2) in the phi = 0 condition:

In [3]:
from civilpy.geotech import shallow_foundation as sf
from civilpy.geotech.spt import undrained_shear_strength_from_n

su = undrained_shear_strength_from_n(8)   # psf, from the clay's N = 8
cap = sf.ultimate_bearing_capacity(
    b=8.0, l=10.0, d=4.0,
    gamma=120.0, phi_deg=0.0, c=su,
    water_table=12.0, gamma_sat=125.0,
)
print(f"Su ~ {su:.0f} psf")
print(f"q_ult  = {cap.q_ult:,.0f} psf")
print(f"q_all  = {cap.q_ult/3.0:,.0f} psf  (FS = 3)")

Su ~ 736 psf
q_ult  = 5,726 psf
q_all  = 1,909 psf  (FS = 3)


## Spread footing — settlement

Schmertmann's strain-influence method for the elastic part (using the SPT
modulus correlation), plus one-dimensional consolidation of the clay
beneath:

In [4]:
from civilpy.geotech.soil_profile import SoilLayer, SoilProfile

profile = SoilProfile([
    SoilLayer("clay (above footing base)", 4.0, 120.0),
    SoilLayer("clay (bearing stratum)", 16.0, 120.0),
    SoilLayer("dense sand", 25.0, 125.0),
], water_table=12.0)

q_net = 2500.0 - 120.0 * 4.0   # net pressure at the 4-ft footing base, psf
consol = sf.consolidation_settlement(
    profile,
    layers=[sf.ConsolidationLayer(layer_index=1, cc=0.25, e0=0.85,
                                  cr=0.05, sigma_pc=2400.0)],
    delta_sigma=lambda z: sf.stress_increase_2to1(q_net, 8.0, 10.0, z - 4.0),
    n_slices=4,
)
print(f"primary consolidation settlement ~ {consol.settlement_in:.2f} in")

primary consolidation settlement ~ 1.09 in


Were the same footing bearing on the *sand*, the settlement would be
elastic and immediate — Schmertmann's strain-influence method with the
Bowles SPT modulus:

In [5]:
from civilpy.geotech.spt import elastic_modulus_from_spt

es_psf = elastic_modulus_from_spt(32, "sand") * 144.0   # psi -> psf
sandy = sf.schmertmann_settlement(
    net_pressure=q_net, b=8.0, l=10.0, d=4.0, es_profile=es_psf,
)
print(f"Es ~ {es_psf/1000:.0f} ksf")
print(f"Schmertmann settlement ~ {sandy.settlement_in:.2f} in "
      f"(C1 = {sandy.c1:.2f}, C2 = {sandy.c2:.2f})")

Es ~ 491 ksf
Schmertmann settlement ~ 0.29 in (C1 = 0.88, C2 = 1.20)


## Driven pile

A 14-inch square displacement pile driven to 40 ft.  `driven_pile_capacity`
walks the boring: alpha·Su side friction and 9·Su end bearing while in the
clay, Meyerhof N60-based friction and tip once it enters the sand.

In [6]:
from civilpy.geotech import deep_foundation as df

b_pile = 14.0 / 12.0
pile = df.driven_pile_capacity(
    boring,
    width_ft=b_pile,
    tip_depth_ft=40.0,
    perimeter_ft=4 * b_pile,       # square section
    area_ft2=b_pile ** 2,
    displacement=True,
)
print(f"side resistance  Rs = {pile.side_nominal_kips:6.1f} kips")
print(f"tip resistance   Rp = {pile.tip_nominal_kips:6.1f} kips")
print(f"nominal capacity Rn = {pile.total_nominal_kips:6.1f} kips")
print(f"factored (AASHTO 10.5.5.2.3 defaults): {pile.factored_kips():.0f} kips")

side resistance  Rs =  145.6 kips
tip resistance   Rp =  368.7 kips
nominal capacity Rn =  514.3 kips
factored (AASHTO 10.5.5.2.3 defaults): 231 kips


## Drilled shaft

A 4-ft shaft tipped in the sand at 42 ft.  Per AASHTO 10.8.3.5 the top 5 ft
and the bottom shaft diameter contribute no side resistance:

In [7]:
shaft = df.drilled_shaft_capacity(boring, diameter_ft=4.0, tip_depth_ft=42.0)
print(f"side resistance  Rs = {shaft.side_nominal_kips:6.1f} kips")
print(f"tip resistance   Rp = {shaft.tip_nominal_kips:6.1f} kips")
print(f"nominal capacity Rn = {shaft.total_nominal_kips:6.1f} kips")
print(f"factored (AASHTO 10.5.5.2.4 defaults): {shaft.factored_kips():.0f} kips")

side resistance  Rs =  485.4 kips
tip resistance   Rp =  482.5 kips
nominal capacity Rn =  967.9 kips
factored (AASHTO 10.5.5.2.4 defaults): 411 kips


For laterally loaded piles, the same boring feeds
`civilpy.geotech.lateral_pile` (p-y curves) and the `civilpy.geotech.lpile`
LPILE file writer/parser — see the *Retaining Wall and Boring Log* notebook
for the earth-pressure side of the story.